# Euler–Bernoulli Beam Problem

#### <p style="text-align: right;"> &#9989; **put your name here** </p>

---
**Recall the course of static mechanics.** 

We solve:

$$ EI \frac{d^4 w}{dx^4} = q $$

with simply supported boundary conditions:

$$ w(0)=0, \quad w(L)=0 $$
$$ w''(0)=0, \quad w''(L)=0 $$

where $w(x)$ is the vertical displacement of the beam, $L$ is the length of the beam. Here, we have assume that there is horizontal displacement of the beam. See the derivations of the equations in [**wikipedia**](https://en.wikipedia.org/wiki/Euler%E2%80%93Bernoulli_beam_theory) if you are interested.

## Part A: Analytical Solution (SymPy)

- Set variables
- Define functions
- Set the equation

In [ ]:
import sympy as sp

# x, E, I, q are varibles
x = sp.symbols('x')
E, I, q = sp.symbols('E I q')

# define w(x)
w = sp.Function('w')

# define the equation: lhs and rhs
ode = sp.Eq(sp.diff(w(x), x, 4), q/(E*I))
sol = sp.dsolve(ode)

sol

### Solve for the integration constants.
- Solve for the integration constants.

In [ ]:
# define constants
C1, C2, C3, C4 = sp.symbols('C1 C2 C3 C4')

# w(x)
w_expr = sol.rhs

# w''(x)
w2 = sp.diff(w_expr, x, 2)
w2

- Use the boundary conditions for solving the constants.

In [ ]:
L = sp.symbols('L')

eq1 = w_expr.subs(x, 0)       # w(0)=0
eq2 = w_expr.subs(x, L)       # w(L)=0
eq3 = w2.subs(x, 0)           # w''(0)=0
eq4 = w2.subs(x, L)           # w''(L)=0

constants = sp.solve([eq1, eq2, eq3, eq4], [C1, C2, C3, C4])
print(constants)

w_final = w_expr.subs(constants)
sp.simplify(w_final)

### Plot the result

- Substitute values for the parameters. Here we simply set their values to be 1.

In [ ]:
# substitute all parameters (using set and dictionary)
w_final = w_final.subs({L:1, q:-1, E:1, I:1})
w_final


- Make the analytical expression to numerical values by using `lambdify`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# convert  w_final to a numpy function that takes x as the variables,
f = sp.lambdify(x, w_final, "numpy")

# x grid
N = 81
xv = np.linspace(0, 1, N)

w_ana = f(xv)

plt.plot(xv, w_ana, 'r.-', label="analytical")
plt.legend()
plt.xlabel('x')
plt.ylabel('deflection w(x)')
plt.title('Beam Deflection (FDM)')
plt.grid()
# plt.show()

## Part B: Finite Difference Method (FDM)

Here, we use the following finite difference stencil (formula) to calculate the values of 4th-order derivatives:
$$\frac{d^4 w}{dx^4} \equiv \frac{w_{i-2}-4w_{i-1} + 6 w_i - 4 w_{i+1} + w_{i+2}}{\Delta x^4}.$$
See [**wikipedia**](https://en.wikipedia.org/wiki/Finite_difference_coefficient) if you're interested in finite difference methods.
For the boundary conditions of the 2nd derivatives, we use
$$\frac{d^2 w}{dx^2} \equiv \frac{w_{i-1} -2 w_i + w_{i+2}}{\Delta x^2}.$$

To the end, we will build a matrix equation $$\mathbf{A}\mathbf{w} =\mathbf{b}$$ based on the stencils, where $\mathbf{w}$ is a column vector containing discrete values of $w_i$.

In [ ]:
import numpy as np

# parameters
L = 1.0
EI = 1.0
q = -1.0

# discretization
N = 81
dx = L/(N-1)
xv = np.linspace(0, 1, N)

# matrix
A = np.zeros((N, N))
b = np.zeros(N)

# interior points
for i in range(2, N-2):
    A[i, i-2] = ??
    A[i, i-1] = ??
    A[i, i]   = ??
    A[i, i+1] = ??
    A[i, i+2] = 1
    b[i] = q * dx**4 / EI

# boundary conditions
# w(0) = 0
A[0,0] = 1
b[0] = 0

# w''(0) = 0
A[1,0] = 1
A[1,1] = ??
A[1,2] = 1
b[1] = 0

# w''(L) = 0
A[-2,-3] = 1
A[-2,-2] = -2
A[-2,-1] = 1
b[-2] = 0

# w(L) = 0
A[-1,-1] = 1
b[-1] = 0

# solve
w_num = np.linalg.solve(A, b)

# plot
plt.plot(xv, w_ana, 'r-', label="analytical")
plt.plot(xv, w_num, 'bo', markerfacecolor='none', label="numerical",)
plt.xlabel('x')
plt.ylabel('deflection w(x)')
plt.title('Beam Deflection (FDM)')
plt.legend()
plt.grid()
plt.show()

## Part C: Questions


&#9989; Do This -- Answer these questions.

1. What does $w''(x)$ represent physically?
2. How does increasing $EI$ affect deflection?


---
## Part D: dynamic beam (vibration)

Now, let's simulation the vibration of the beam. For dynamic problems, the force balance is based on the Newon's Law: $$f = ma,$$ where $a = \ddot{w}$ is the vertical acceleration. Thus, the governing equation becomes

$$- \mu \frac{\partial^2 w}{\partial t^2} = EI \frac{\partial^4 w}{\partial x^4} + q(x).$$

Here, we have a 2nd derivative in time. We can use the following stencil for point $i$:

$$-\mu \frac{w_i^{n+1} -2 w_i^{n} + w_i^{n-1}}{\Delta t} = \frac{w_{i-2}-4w_{i-1} + 6 w_i - 4 w_{i+1} + w_{i+2}}{\Delta x^4} + q_i$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output

# -----------------------------
# Dynamic Euler-Bernoulli beam
# mu w_tt + EI w_xxxx = q(x,t)
# -----------------------------

# parameters
L = 1.0
EI = 1.0      # bending stiffness
mu = 75.0      # mass per unit length
q = -1

# discretization
N = 80
dx = L / (N - 1)
xv = np.linspace(0, L, N)

# time step: explicit scheme needs small dt
dt = 1e-4 
nt = 150001

# initial condition
w_old = np.zeros(N)
w = 0.01 * np.sin(np.pi * xv / L)   # initial deflection
w_new = np.zeros(N)
w_old = np.copy(w)

# coefficient
coef = dt**2/mu

print(coef)

# store center deflection
center_history = []


for n in range(nt):
    t = n * dt

    # update interior points
    for i in range(2, N-2):
        w_xxxx = (??)/dx**4
        w_new[i] = ??

    # simply supported boundary conditions:
    # w = 0 at both ends
    w_new[0] = 0
    w_new[-1] = 0

    # approximate w'' = 0 at both ends
    w_new[1] = (w_new[0] + w_new[2])/2
    w_new[-2] = (w_new[-3] + w_new[-1])/2
    
    # update time levels
    w_old[:] = w
    w[:] = w_new

    center_history.append(w[N//2])

    # plot occasionally
    if n % 1000 == 0:
        clear_output(wait=True)
        plt.figure(figsize=(7,4))
        plt.plot(xv, w, 'b-', lw=2)
        plt.ylim(-0.05, 0.05)
        plt.xlabel("x")
        plt.ylabel("w(x,t)")
        plt.title(f"Dynamic Euler-Bernoulli Beam, t = {t:.4f}")
        plt.grid(True)
        plt.show()

# plot center deflection history
time = np.arange(len(center_history)) * dt

plt.figure(figsize=(7,4))
plt.plot(time, center_history, 'r-')
plt.xlabel("time")
plt.ylabel("center deflection")
plt.title("Oscillation of beam midpoint")
plt.grid(True)
plt.show()

print(coef)

## Part E: further exploration

- Now, assuming a weight of 4 is hung at approximately $x = 0.25$. Modify the code to simulate this scenario. 
- Chance the initial configuration of the beam and simulate the beam dynamics.


In [ ]:
# find the point where the load is
idx = int(0.25*80)
print(idx)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output

# -----------------------------
# Dynamic Euler-Bernoulli beam
# mu w_tt + EI w_xxxx = q(x,t)
# -----------------------------

# parameters
L = 1.0
EI = 1.0      # bending stiffness
mu = 75.0      # mass per unit length
q = -1

# discretization
N = 80
dx = L / (N - 1)
xv = np.linspace(0, L, N)

# time step: explicit scheme needs small dt
dt = 1e-4 
nt = 150001

# initial condition
w_old = np.zeros(N)
w = 0.01 * np.sin(np.pi * xv *2 / L)   # initial deflection
w_new = np.zeros(N)
w_old = np.copy(w)

# coefficient
coef = dt**2/mu

print(coef)


# store center deflection
center_history2 = []

wt = 4
for n in range(nt):
    t = n * dt

    # update interior points
    for i in range(2, N-2):
        w_xxxx = ??
        if i == idx:
            w_new[i] = 2*w[i] - w_old[i] - ??
        else:
            w_new[i] = 2*w[i] - w_old[i] - ??

    # simply supported boundary conditions:
    # w = 0 at both ends
    w_new[0] = 0
    w_new[-1] = 0

    # approximate w'' = 0 at both ends
    w_new[1] = (w_new[0] + w_new[2])/2
    w_new[-2] = (w_new[-3] + w_new[-1])/2
    
    # update time levels
    w_old[:] = w
    w[:] = w_new

    center_history2.append(w[N//2])

    # plot occasionally
    if n % 1000 == 0:
        clear_output(wait=True)
        plt.figure(figsize=(7,4))
        plt.plot(xv, w, 'b-', lw=2)
        plt.ylim(-0.05, 0.05)
        plt.xlabel("x")
        plt.ylabel("w(x,t)")
        plt.title(f"Dynamic Euler-Bernoulli Beam, t = {t:.4f}")
        plt.grid(True)
        plt.show()

# plot center deflection history
time = np.arange(len(center_history)) * dt

plt.figure(figsize=(7,4))
plt.plot(time, center_history2, 'g-')
plt.xlabel("time")
plt.ylabel("center deflection")
plt.title("Oscillation of beam midpoint")
plt.grid(True)
plt.show()

print(coef)

In [ ]:

plt.figure(figsize=(7,4))
plt.plot(time, center_history, 'r-')
plt.plot(time, center_history2, 'g-')
plt.xlabel("time")
plt.ylabel("center deflection")
plt.title("Oscillation of beam midpoint")
plt.grid(True)
plt.show()

## You're done with this module. 
Please upload your file via this [**link**](https://www.dropbox.com/request/mbfdaf1mof70lglc9qqr).